# 🦷 DentalPilot AI - 口腔影像模型训练笔记本

**用途**：用DENTEX公开数据集训练YOLOv8牙齿检测+病灶识别模型

**运行环境**：Google Colab（免费T4 GPU）

**预计时间**：
- 训练牙齿检测：1-2小时
- 训练病灶识别：2-3小时

**输出**：
- `models/yolov8n_teeth.pt` 牙齿检测权重
- `models/yolov8n_disease.pt` 病灶识别权重
- 上传到 Hugging Face Model Hub（公开/私有）

## 第一步：环境准备

In [ ]:
# 安装依赖
!pip install -q ultralytics==8.1.0
!pip install -q roboflow  # 用于下载DENTEX数据集
!pip install -q albumentations  # 数据增强
!pip install -q opencv-python-headless

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 第二步：克隆代码仓库

In [ ]:
# 克隆你的GitHub仓库（首次使用）
# 后续可以直接 !cd dental-ai && !git pull
import os

REPO_URL = "https://github.com/air199009/pin-guan-oral.git"  # 替换成你的仓库

if not os.path.exists("pin-guan-oral"):
    !git clone {REPO_URL}
    print("✅ 仓库克隆完成")
else:
    %cd pin-guan-oral
    !git pull
    print("✅ 代码已更新")

%cd pin-guan-oral
!mkdir -p models data/datasets

## 第三步：下载DENTEX数据集

In [ ]:
# 方法1：从Roboflow下载DENTEX（推荐，自动转YOLO格式）
from roboflow import Roboflow

# 公开数据集（无需API key）
rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY")  # 免费注册 https://roboflow.com 获取
project = rf.workspace("dental-ai").project("dentex-2024")
dataset = project.version(1).download("yolov8")

print(f"✅ 数据集已下载到: {dataset.location}")

In [ ]:
# 方法2：使用其他公开数据集
# 备选：UFBA-UESC牙齿实例分割数据集
!wget -q https://github.com/ricoleehduu/STS-Challenge-2024/releases/download/v1.0/dataset.zip -O /tmp/sts.zip
!unzip -q /tmp/sts.zip -d data/datasets/sts2024/
print("✅ STS2024 数据集已下载")
print("   包含：90,000+ 2D全景X光 + 330 CBCT")

## 第四步：训练牙齿检测模型（YOLOv8n）

In [ ]:
from ultralytics import YOLO

# 加载预训练模型（迁移学习，比从0开始快10倍）
model = YOLO("yolov8n.pt")

# 训练配置
results = model.train(
    data="data/datasets/dentex/data.yaml",  # 数据集配置（自动生成）
    epochs=50,                # 训练轮数（首次可以50，迭代时100-200）
    imgsz=640,                # 图片尺寸
    batch=16,                 # 批次（GPU小可调到8）
    name="teeth_detector",
    project="runs",
    device=0,                 # GPU 0
    workers=2,
    patience=20,              # 早停
    save=True,
    # 数据增强
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    # 优化器
    optimizer="AdamW",
    lr0=1e-3,
    cos_lr=True,
)

print("\n" + "="*60)
print("✅ 训练完成")
print("="*60)

In [ ]:
# 查看训练结果
from IPython.display import Image as IPImage, display
import glob

# 训练曲线
results_img = glob.glob("runs/teeth_detector/results.png")
if results_img:
    print("📊 训练曲线：")
    display(IPImage(results_img[0], width=800))

# 验证集预测样例
val_img = glob.glob("runs/teeth_detector/val_batch0_pred.jpg")
if val_img:
    print("\n🎯 验证集预测样例：")
    display(IPImage(val_img[0], width=800))

In [ ]:
# 复制最佳权重到 models/ 目录
!cp runs/teeth_detector/weights/best.pt models/yolov8n_teeth.pt
print("✅ 牙齿检测模型已保存: models/yolov8n_teeth.pt")

# 验证模型
model = YOLO("models/yolov8n_teeth.pt")
metrics = model.val()
print(f"\n📈 模型指标：")
print(f"   mAP50: {metrics.box.map50:.3f}")
print(f"   mAP50-95: {metrics.box.map:.3f}")
print(f"   Precision: {metrics.box.mp:.3f}")
print(f"   Recall: {metrics.box.mr:.3f}")

## 第五步：训练病灶识别模型（可选）

**注意**：需要自己有标注数据，或使用DENTEX challenge 2024的异常标注

In [ ]:
# 病灶识别模型（如果有标注数据）
disease_model = YOLO("yolov8n.pt")

if os.path.exists("data/datasets/disease_yolo/data.yaml"):
    disease_results = disease_model.train(
        data="data/datasets/disease_yolo/data.yaml",
        epochs=100,
        imgsz=640,
        batch=16,
        name="disease_detector",
        project="runs",
        patience=30,
    )
    !cp runs/disease_detector/weights/best.pt models/yolov8n_disease.pt
    print("✅ 病灶识别模型已保存")
else:
    print("⚠️ 病灶数据集未准备好，跳过此步")
    print("   可以用 DENTEX challenge 2024 或自标注数据训练")

## 第六步：测试模型效果

In [ ]:
# 用测试图片测试
from PIL import Image
import os

model = YOLO("models/yolov8n_teeth.pt")

# 用DENTEX验证集的一张图片测试
test_imgs = glob.glob("data/datasets/dentex/test/images/*.jpg")[:5]
if test_imgs:
    for img_path in test_imgs:
        results = model(img_path, save=True, conf=0.3)
        print(f"✅ 测试: {os.path.basename(img_path)}")
        
    # 显示结果
    result_img = glob.glob("runs/detect/predict/*.jpg")
    if result_img:
        display(IPImage(result_img[0], width=600))

## 第七步：上传模型到 Hugging Face

In [ ]:
# 上传到 Hugging Face Model Hub
!pip install -q huggingface_hub

from huggingface_hub import HfApi, login

# 登录（需要先注册：https://huggingface.co）
HF_TOKEN = "YOUR_HF_TOKEN"  # 在 https://huggingface.co/settings/tokens 获取
HF_USERNAME = "air199009"  # 你的HF用户名

login(token=HF_TOKEN)
api = HfApi()

# 创建模型仓库
repo_id = f"{HF_USERNAME}/dentalpilot-yolov8-teeth"
api.create_repo(repo_id=repo_id, exist_ok=True, private=False)

# 上传权重
api.upload_file(
    path_or_fileobj="models/yolov8n_teeth.pt",
    path_in_repo="yolov8n_teeth.pt",
    repo_id=repo_id,
)

print(f"✅ 模型已上传: https://huggingface.co/{repo_id}")

## 第八步：导出 ONNX（用于本地部署加速）

In [ ]:
# 导出为 ONNX 格式（部署用，更快）
model = YOLO("models/yolov8n_teeth.pt")
model.export(format="onnx", imgsz=640, simplify=True)

!ls -lh models/
print("\n✅ ONNX 导出完成")
print("   部署到客户本地时使用 .onnx 文件，推理速度提升 2-3 倍")

## 第九步：提交到GitHub

In [ ]:
# 提交到 GitHub
!git add models/
!git config user.email "your@email.com"
!git config user.name "DentalPilot"
!git commit -m "feat: trained teeth detection model v1.0"
!git push origin main

print("✅ 已推送到 GitHub")
print(f"   仓库地址: {REPO_URL}")

---

## 🎉 完成！

**下一步**：

1. **本地测试**：回到你电脑，激活虚拟环境，运行 `python app.py`
2. **浏览器访问**：打开 http://127.0.0.1:7860
3. **上传测试影像**：体验完整流程
4. **约诊所演示**：约1家信任的诊所老板现场演示
5. **收集反馈**：用 Tab 2 收集医生修正意见
6. **持续训练**：每周重新训练一次，模型会越用越好

**遇到问题？**
- 看 `docs/客户部署手册.md`
- 在 GitHub 提 Issue
- 联系开发者：你的微信号/邮箱
